# Role Generation for Spider Database Tables

This notebook performs role-based access control (RBAC) analysis for the Spider database collection using LLM.

### 0. Import lib and env

In [5]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path('/home/feiy/Role-SQL-benchmark')
sys.path.append(str(project_root))

# Import required modules
from src.role_parser import RoleGenerator, ParallelRoleGenerator
from src.processors.sql_data_process import SpiderDataProcessor
from src.processors.role_sql_generate import RoleSQLGenerator
from dotenv import load_dotenv
import os
import json
from datetime import datetime
import logging
import random
import importlib
import src.llm_oracle as oracle

# Setup global timestamp for this run
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# Setup output directories
log_dir = project_root / 'logs'
output_dir = project_root / 'outputs'

for directory in [log_dir, output_dir]:
    directory.mkdir(exist_ok=True)

# Configure logging
log_file = log_dir / f'role_assignment_{RUN_TIMESTAMP}.log'
logging.getLogger().handlers.clear()

logger = logging.getLogger('role_assignment')
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(str(log_file))
console_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

for handler in [file_handler, console_handler]:
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.propagate = False
logger.info(f"Starting new session at {RUN_TIMESTAMP}")
logger.info(f"Log file: {log_file}")
logger.info(f"Output directory: {output_dir}")

2025-09-18 22:55:15,210 - INFO - Starting new session at 20250918_225515
2025-09-18 22:55:15,211 - INFO - Log file: /home/feiy/Role-SQL-benchmark/logs/role_assignment_20250918_225515.log
2025-09-18 22:55:15,211 - INFO - Output directory: /home/feiy/Role-SQL-benchmark/outputs
2025-09-18 22:55:15,211 - INFO - Log file: /home/feiy/Role-SQL-benchmark/logs/role_assignment_20250918_225515.log
2025-09-18 22:55:15,211 - INFO - Output directory: /home/feiy/Role-SQL-benchmark/outputs


In [ ]:
# Load environment variables and API keys
load_dotenv()
importlib.reload(oracle)

In [ ]:
# DeepSeek demo
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not DEEPSEEK_API_KEY:
    print("Warning: Cannot find DEEPSEEK_API_KEY in environment")
    print("Please set it in the .env file or environment")
if not OPENAI_API_KEY:
    print("Warning: Cannot find OPENAI_API_KEY in environment")
    print("Please set it in the .env file or environment")


### 1. Setup LLM Oracle instance

In [ ]:
def create_oracle(model_name, api_key):
    return oracle.Oracle(model_name, api_key)

#### 1.1. Deepseek test

##### 1.1(a) deepseek demo test

In [ ]:
# if not DEEPSEEK_API_KEY:
#     print("Warning: DEEPSEEK_API_KEY not found in environment variables")
#     print("Please set it in your .env file or environment")
# else:
#     # Create Oracle instance with DeepSeek model
#     oracle = Oracle(model="deepseek-chat", apikey=DEEPSEEK_API_KEY)
    
#     # Test the model
#     test_response = oracle.query(
#         prompt_sys="You are a helpful assistant.",
#         prompt_user="Say Hi.",
#         temp=0.7,
#         top_p=0.9
#     )
    
#     print("Model Response:")
#     print("="*50)
#     print(test_response['answer'])

##### 1.1(b) prompt caching test

In [ ]:
# # Create two identical requests to test caching
# prompt_sys = """You are a helpful assistant. You should:
# 1. Be concise and clear in your responses
# 2. Always strive to provide accurate information
# 3. Maintain a professional and friendly tone
# 4. Use appropriate formatting when needed
# 5. Ask for clarification if something is unclear"""

# prompt_user = """Please introduce yourself and tell me about your capabilities.
# Make sure to mention:
# 1. Your name
# 2. Your main areas of expertise
# 3. How you can help users
# 4. Any limitations users should be aware of"""

In [ ]:
# def make_query(prompt_sys, prompt_user):
#     """Execute query and return response"""
#     response = oracle.query(
#         prompt_sys=prompt_sys,
#         prompt_user=prompt_user,
#     )
#     # if response.get('answer'):
#     #     print(f"Response: {response['answer']}")
#     return response

# def print_cache_stats(response, label=None):
#     """Print cache statistics for a response"""
#     if label:
#         print(f"\n=== {label} Statistics ===")
    
#     usage = response.get('usage', {})
#     prompt_details = usage.get('prompt_tokens_details', None)
    
#     print("\nToken Statistics:")
#     print(f"  Prompt tokens: {usage.get('prompt_tokens', 0)}")
#     print(f"  Completion tokens: {usage.get('completion_tokens', 0)}")
#     print(f"  Total tokens: {usage.get('total_tokens', 0)}")
    
#     print("\nCache Statistics:")
#     if prompt_details:
#         if isinstance(prompt_details, str):
#             print(f"  Raw info: {prompt_details}")
#         else:
#             cached = getattr(prompt_details, 'cached_tokens', 0)
#             non_cached = usage.get('prompt_tokens', 0) - cached
#             print(f"  Cached tokens: {cached}")
#             print(f"  Non-cached tokens: {non_cached}")
#             if cached > 0:
#                 print(f"  Cache hit rate: {(cached / usage.get('prompt_tokens', 1)) * 100:.1f}%")

In [ ]:
# response1 = make_query(prompt_sys, prompt_user)
# response2 = make_query(prompt_sys, prompt_user)
# response3 = make_query(prompt_sys, prompt_user)

# print_cache_stats(response1, "First Call")
# print_cache_stats(response2, "Second Call")
# print_cache_stats(response3, "Third Call")

#### 1.2. OpenAI Test

In [ ]:
# Example: create Oracle instance (model and api_key should be set according to your environment)

# MODEL_NAME = 'gpt-4o'  # or any supported model
# create_oracle_instance = create_oracle(model_name=MODEL_NAME, api_key=OPENAI_API_KEY)
# print(f"Oracle instance created for model: {MODEL_NAME}")

In [ ]:
# build a simple prompt for testing

# prompt_sys = """You are a helpful assistant. You should:
# 1. Be concise and clear in your responses
# 2. Always strive to provide accurate information
# 3. Maintain a professional and friendly tone
# 4. Use appropriate formatting when needed
# 5. Ask for clarification if something is unclear"""
# prompt_user = """Please introduce yourself and tell me about your capabilities.
# Make sure to mention:
# 1. Your name
# 2. Your main areas of expertise
# 3. How you can help users
# 4. Any limitations users should be aware of"""
# response = openai_oracle_instance.query(
#     prompt_sys=prompt_sys,
#     prompt_user=prompt_user,
# )
# print(f"Response: {response['answer']}")

### 2. Role Assignment for Spider Database Tables

This section aims to:
1. Read schema information from Spider database
2. Use LLM to generate appropriate roles for each table

#### 2.1 Overview for Spider dataset
This Part will:
1. Basic stats for Spider Database
2. Process and reformat spider file to training dataset

In [ ]:
# Initialize processor with new flexible architecture
processor = SpiderDataProcessor(project_root)

# Get all database folders and statistics
db_folders = processor.get_db_folders()
test_db_folders = processor.get_test_db_folders()
db_stats = processor.get_db_statistics(is_test=False)
test_db_stats = processor.get_db_statistics(is_test=True)

# Calculate and log statistics for training&dev dataset
logger.info("\nSpider Database Statistics:")
logger.info("-" * 40)

total_dbs = len(db_stats)
dbs_with_sqlite = sum(1 for stats in db_stats.values() if stats['has_sqlite'])
dbs_with_schema = sum(1 for stats in db_stats.values() if stats['has_schema'])
total_tables = sum(stats['table_count'] for stats in db_stats.values())

logger.info(f"Found {len(db_folders)} databases in Spider Training&Dev dataset")
logger.info(f"Total databases in statistics: {total_dbs}")
logger.info(f"Databases with SQLite files: {dbs_with_sqlite}")
logger.info(f"Databases with schema files: {dbs_with_schema}")
logger.info(f"Total tables across all databases: {total_tables}")
logger.info(f"Average tables per database: {total_tables/dbs_with_sqlite:.2f}")

# Calculate and log statistics for test dataset
logger.info(f"\nTest Database Statistics:")
logger.info("-" * 40)

total_test_dbs = len(test_db_stats)
test_dbs_with_sqlite = sum(1 for stats in test_db_stats.values() if stats['has_sqlite'])
test_dbs_with_schema = sum(1 for stats in test_db_stats.values() if stats['has_schema'])
test_total_tables = sum(stats['table_count'] for stats in test_db_stats.values())

# 计算平均值（避免除零错误）
avg_tables_per_db = f"{test_total_tables/test_dbs_with_sqlite:.2f}" if test_dbs_with_sqlite else "0.00"

logger.info(f"Found {len(test_db_folders)} databases in Spider Test dataset")
logger.info(f"Total databases in statistics: {total_test_dbs}")
logger.info(f"Databases with SQLite files: {test_dbs_with_sqlite}")
logger.info(f"Databases with schema files: {test_dbs_with_schema}")
logger.info(f"Total tables across all databases: {test_total_tables}")
logger.info(f"Average tables per database: {avg_tables_per_db}")

# Additional Spider dataset information
logger.info("\nDetailed database statistics have been saved to spider_info.json and spider_test_info.json")
logger.info("You can find them in the data directory")

# Process all Spider data using flexible architecture
logger.info("\nProcessing Spider Data with Flexible Architecture:")
logger.info("-" * 55)
logger.info("Processing train, dev, test, and combined datasets...")

try:
    # Check if new methods are available
    if hasattr(processor, 'process_all_spider_data'):
        # Use new flexible data processing
        results = processor.process_all_spider_data()
        
        logger.info("Successfully processed all Spider datasets:")
        for source, count in results.items():
            logger.info(f"  - {source}: {count:,} examples")
    else:
        # Fallback to individual processing
        logger.info("Using individual processing methods:")
        
        # Process train (including dev for compatibility)
        train_result = processor.process_spider_train_data(include_dev=True)
        logger.info(f"  - train (with dev): processed successfully")
        
        logger.info("✅ Data processing completed using fallback method")
    
except Exception as e:
    logger.error(f"Error processing Spider data: {str(e)}")
    logger.error("Please check if train_spider.json, dev.json exist and are accessible")

#### 2.2 Prompt Design and Role Assignment

The part is designed to (refer to configs/prompts.py for detailed prompt):
1. Provide clear context about the task (Role-Based Access Control)
2. Guide the LLM to analyze table schema and relationships
3. Generate appropriate role names and descriptions
4. Maintain consistency across different tables

In [ ]:
# Process all databases in batches
BATCH_SIZE = 24
N_WORKERS = 20

In [ ]:
# Helper function for processing databases in batches
def process_databases_batch(db_folders, is_test=False):
    # Filter databases that have schema files
    valid_dbs = [db for db in db_folders if (db / "schema.sql").exists()]
    total_dbs = len(valid_dbs)
    n_batches = (total_dbs + BATCH_SIZE - 1) // BATCH_SIZE  # Ceiling division

    dataset_type = "test" if is_test else "training and dev"
    logger.info(f"\nStarting batch processing for all databases in spider {dataset_type} database:")
    logger.info(f"Total valid databases: {total_dbs}")
    logger.info(f"Batch size: {BATCH_SIZE}")
    logger.info(f"Number of batches: {n_batches}")
    logger.info(f"Workers per batch: {N_WORKERS}")

    # Initialize role generator (reused for all batches)
    generator = ParallelRoleGenerator(model="deepseek-chat", api_key=DEEPSEEK_API_KEY, n_workers=N_WORKERS)
    logger.info(f"Initialized ParallelRoleGenerator with model: deepseek-chat")

    # Process each batch
    all_role_assignments = {}
    total_processed = 0
    total_roles = 0

    for batch_idx in range(n_batches):
        batch_start = batch_idx * BATCH_SIZE
        batch_end = min(batch_start + BATCH_SIZE, total_dbs)
        batch_dbs = valid_dbs[batch_start:batch_end]
        
        logger.info(f"\nProcessing Batch {batch_idx + 1}/{n_batches}")
        logger.info(f"Databases in this batch: {[db.name for db in batch_dbs]}")
        
        # Process batch in parallel
        sqlite_paths = {db.name: str(db / f"{db.name}.sqlite") for db in batch_dbs}
        results = generator.process_databases_parallel(batch_dbs, sqlite_paths=sqlite_paths)
        
        # Process results from this batch
        batch_processed = 0
        batch_roles = 0
        
        for result in results:
            if result and result.get('roles'):
                batch_processed += 1
                roles_count = len(result['roles'])
                batch_roles += roles_count
                all_role_assignments[result['database']] = result['roles']
            else:
                logger.error(f"Failed to process one of the databases in batch {batch_idx + 1}")
        
        # Update totals
        total_processed += batch_processed
        total_roles += batch_roles
        
        # Log batch results
        logger.info(f"Batch {batch_idx + 1} completed:")
        logger.info(f"- Databases processed in this batch: {batch_processed}/{len(batch_dbs)}")
        logger.info(f"- Roles generated in this batch: {batch_roles}")
        logger.info(f"- Total progress: {total_processed}/{total_dbs} databases processed")

    # Prepare final metadata
    assignments_data = {
        'assignments': all_role_assignments,
        'metadata': {
            'timestamp': RUN_TIMESTAMP,
            'total_databases': total_dbs,
            'processed_databases': total_processed,
            'total_roles_generated': total_roles,
            'batch_size': BATCH_SIZE,
            'n_workers': N_WORKERS,
            'n_batches': n_batches,
            'dataset_type': dataset_type
        }
    }

    # Save all results
    if all_role_assignments:
        suffix = '_test' if is_test else '_train'
        output_file = generator.save_assignments_parallel(assignments_data, output_dir, f"{RUN_TIMESTAMP}{suffix}")

    logger.info(f"\nAll {dataset_type} batches completed:")
    logger.info(f"- Total databases processed: {total_processed}/{total_dbs}")
    logger.info(f"- Total roles generated: {total_roles}")
    logger.info(f"- Average roles per database: {total_roles/total_processed if total_processed else 0:.2f}")
    
    return assignments_data

In [ ]:
# Process training and dev databases
logger.info("\n" + "="*50)
logger.info("Processing Training & Dev Databases")
logger.info("="*50)
train_assignments = process_databases_batch(db_folders, is_test=False)

# Process test databases
logger.info("\n" + "="*50)
logger.info("Processing Test Databases")
logger.info("="*50)
test_assignments = process_databases_batch(test_db_folders, is_test=True)

### 3. Generate Role-Based SQL Dataset

Generate role-based text2sql dataset by combining Spider data with role assignments.

In [2]:
# Helper function to generate role-SQL dataset for given data source and role assignments
def generate_role_sql_dataset_with_stats(generator, source, role_file_path):
    logger.info(f"\nGenerating Role-SQL dataset for {source.upper()} data:")
    logger.info("-" * 50)
    
    # Generate the dataset using the correct API
    dataset = generator.generate_role_sql_dataset(role_file_path=role_file_path, data_source=source)
    
    if isinstance(dataset, list) and len(dataset) > 0:
        # Calculate statistics
        total_examples = len(dataset)
        databases = len({example['db_id'] for example in dataset if isinstance(example, dict)})
        roles = len({(example['db_id'], example['role']) for example in dataset if isinstance(example, dict)})
        denied_queries = sum(1 for example in dataset if isinstance(example, dict) and "Sorry, I cannot answer." in str(example.get('query', '')))
        
        logger.info(f"{source.upper()} Dataset Statistics:")
        logger.info(f"  Total examples: {total_examples:,}")
        logger.info(f"  Databases: {databases}")
        logger.info(f"  Unique roles: {roles}")
        logger.info(f"  Denied queries: {denied_queries} ({denied_queries/total_examples*100:.2f}%)")
    else:
        logger.warning(f"  {source} dataset is empty or invalid")
        
    return dataset

In [4]:
# Initialize generator
generator = RoleSQLGenerator(project_root=project_root)

# Display available data sources
logger.info("\nAvailable data sources for Role-SQL generation:")
logger.info("-" * 50)
for source, path in generator.data_sources.items():
    if path.exists():
        data = generator.load_spider_data(source)
        logger.info(f"  {source}: {len(data):,} examples ({path.name})")
    else:
        logger.info(f"  {source}: Data file not found")

# Get the role assignments files for both train and test data
train_role_file = str(project_root / f'outputs/role_assignments_{RUN_TIMESTAMP}_train.json')
test_role_file = str(project_root / f'outputs/role_assignments_{RUN_TIMESTAMP}_test.json')

# Generate datasets for train and test
all_datasets = {}

# Process training data
if os.path.exists(train_role_file):
    logger.info("\nProcessing training and dev data...")
    train_dataset = generate_role_sql_dataset_with_stats(generator, 'train', train_role_file)
    dev_dataset = generate_role_sql_dataset_with_stats(generator, 'dev', train_role_file)
    all_datasets['train'] = train_dataset
    all_datasets['dev'] = dev_dataset
else:
    logger.warning(f"Training role assignments file not found: {train_role_file}")

# Process test data
if os.path.exists(test_role_file):
    logger.info("\nProcessing test data...")
    test_dataset = generate_role_sql_dataset_with_stats(generator, 'test', test_role_file)
    all_datasets['test'] = test_dataset
else:
    logger.warning(f"Test role assignments file not found: {test_role_file}")

# Generate combined dataset if we have both train and test data
if len(all_datasets) > 1:
    logger.info("\nCreating combined dataset...")
    try:
        combined_data = []
        for source, dataset in all_datasets.items():
            if isinstance(dataset, list):
                for item in dataset:
                    if isinstance(item, dict):
                        item['source'] = source
                        combined_data.append(item)
        
        all_datasets['combined'] = combined_data
        
        total_examples_all = len(combined_data)
        logger.info(f"\nCombined Dataset Statistics:")
        logger.info("-" * 35)
        logger.info(f"Total examples: {total_examples_all:,}")
        logger.info(f"Databases: {len({ex['db_id'] for ex in combined_data})}")
        logger.info(f"Unique roles: {len({(ex['db_id'], ex['role']) for ex in combined_data})}")
    except Exception as e:
        logger.error(f"Error creating combined dataset: {str(e)}")

# Save individual datasets with appropriate suffixes
for source, dataset in all_datasets.items():
    if dataset:  # Only save non-empty datasets
        output_path = str(project_root / f'outputs/role_sql_dataset_{RUN_TIMESTAMP}_{source}.json')
        try:
            if hasattr(generator, 'save_dataset'):
                generator.save_dataset(dataset, output_path=output_path)
            else:
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(dataset, f, ensure_ascii=False, indent=2)
            logger.info(f"\n{source.upper()} dataset saved to: {output_path}")
        except Exception as e:
            logger.error(f"Error saving {source} dataset: {str(e)}")

logger.info("\n Role-SQL dataset generation completed for all data sources!")

2025-09-18 22:47:54,156 - INFO - 
Available data sources for Role-SQL generation:
2025-09-18 22:47:54,157 - INFO - --------------------------------------------------
2025-09-18 22:47:54,157 - INFO - --------------------------------------------------
2025-09-18 22:47:54,174 - INFO -   train: 7,000 examples (spider_train_data.json)
2025-09-18 22:47:54,180 - INFO -   dev: 1,034 examples (spider_dev_data.json)
2025-09-18 22:47:54,185 - INFO -   test: 2,147 examples (spider_test_data.json)
2025-09-18 22:47:54,174 - INFO -   train: 7,000 examples (spider_train_data.json)
2025-09-18 22:47:54,180 - INFO -   dev: 1,034 examples (spider_dev_data.json)
2025-09-18 22:47:54,185 - INFO -   test: 2,147 examples (spider_test_data.json)
2025-09-18 22:47:54,200 - INFO -   combined: 10,181 examples (spider_combined_data.json)
2025-09-18 22:47:54,201 - INFO - 
Processing training and dev data...
2025-09-18 22:47:54,201 - INFO - 
Generating Role-SQL dataset for TRAIN data:
2025-09-18 22:47:54,202 - INFO - 

#### 3.1 Flexible Data Source Selection

The new architecture supports generating role-SQL datasets for different data sources:
- **train**: Training data (7,000 examples from 140 databases)
- **dev**: Development/validation data (1,034 examples from 20 databases) 
- **test**: Test data (2,147 examples from 40 databases)
- **combined**: All data combined (10,181 examples from 206 databases)

This allows for flexible experimentation with different dataset sizes and database distributions.

In [ ]:
# Optional: Generate for specific data source only
# Uncomment the following lines to generate only for specific sources

# Example 1: Generate only for dev dataset (smaller, faster for testing)
# logger.info("Generating Role-SQL dataset for DEV data only:")
# dev_dataset = generator.generate_role_sql_dataset('dev', role_file_path=role_assignments_file)
# logger.info(f"Generated {len(dev_dataset):,} examples for dev dataset")

# Example 2: Generate only for train dataset (main training data)
# logger.info("Generating Role-SQL dataset for TRAIN data only:")
# train_dataset = generator.generate_role_sql_dataset('train', role_file_path=role_assignments_file)  
# logger.info(f"Generated {len(train_dataset):,} examples for train dataset")

# Example 3: Process specific sources individually
# sources_to_process = ['train', 'dev']  # Customize as needed
# individual_results = {}
# for source in sources_to_process:
#     logger.info(f"Processing {source} dataset...")
#     individual_results[source] = generator.generate_role_sql_dataset(source, role_file_path=role_assignments_file)
#     logger.info(f"  Generated {len(individual_results[source]):,} examples")

logger.info("✅ Flexible architecture is ready - uncomment above code for individual dataset generation")

In [ ]:
# # Sample some examples from the dataset
# def print_example(example):
#     print(f"Database: {example['db_id']}")
#     print(f"Role: {example['role']}")
#     print(f"Tables accessible: {example['tables']}")
#     print(f"Question: {example['question']}")
#     print(f"Query: {example['query']}")
#     print("-" * 80)

# # Sample and print 5 random examples
# print("Sample Dataset Examples:")
# print("=" * 80)
# for example in random.sample(dataset, min(5, len(dataset))):
#     print_example(example)